# Airline Satisfaction Classification
Predict whether an airline passenger is satisfied or dissatisfied.

## Project Overview
Binary classification on airline passenger survey data (~130K rows). Predict satisfaction from demographics, flight details, and 14 service ratings.

## Learning Objectives
- Work with large survey datasets
- Handle ordinal rating features
- Build and compare models for customer satisfaction prediction

## Problem Statement
Given passenger demographics, flight details, and service ratings, predict whether the passenger is satisfied or dissatisfied.

## Why This Project Matters
Understanding satisfaction drivers helps airlines prioritize service improvements, reduce churn, and increase revenue.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Kaggle: teejmahal20/airline-passenger-satisfaction |
| **Target** | satisfaction (binary) |
| **Rows** | ~130K |
| **Features** | 22 (demographics + flight + 14 service ratings) |

## Dataset Source & License
Kaggle: CC0 Public Domain. Synthetic/anonymized airline survey data.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

## Imports

In [ ]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [ ]:
MAX_ROWS = 50000

## Dataset Loading

In [ ]:
import kagglehub
path = kagglehub.dataset_download('teejmahal20/airline-passenger-satisfaction')
import glob
csvs = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
print('Files:', csvs)
# Use the largest CSV (train set)
csvs.sort(key=lambda x: os.path.getsize(x), reverse=True)
df = pd.read_csv(csvs[0])
df.columns = df.columns.str.strip()
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED)
print(f'Shape: {df.shape}')
df.head()

## Data Validation

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

# Find target
sat_cols = [c for c in df.columns if 'satisfaction' in c.lower()]
TARGET = sat_cols[0] if sat_cols else df.columns[-1]
print(f'\nTarget: {TARGET}')
print(df[TARGET].value_counts())

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#e74c3c','#2ecc71'], edgecolor='black')
axes[0,0].set_title('Satisfaction Distribution')

num_cols = df.select_dtypes('number').columns.tolist()
num_cols = [c for c in num_cols if c != TARGET and 'id' not in c.lower() and 'unnamed' not in c.lower()]
for i, col in enumerate(num_cols[:3]):
    ax = axes[(i+1)//2, (i+1)%2]
    for label in df[TARGET].unique()[:2]:
        subset = df[df[TARGET] == label]
        ax.hist(subset[col].dropna(), bins=30, alpha=0.5, label=str(label), edgecolor='black')
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [ ]:
print(df[TARGET].value_counts())
print(f'\nPositive rate: {df[TARGET].value_counts().iloc[-1]/len(df):.2%}')

## Train/Test Split

In [ ]:
# Encode target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df[TARGET] = le.fit_transform(df[TARGET])
print(f'Classes: {le.classes_}')

# Drop ID columns
drop_cols = [c for c in df.columns if 'id' in c.lower() or 'unnamed' in c.lower()]
df = df.drop(columns=drop_cols, errors='ignore')

# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    df[c] = df[c].astype('category').cat.codes

df = df.fillna(df.median(numeric_only=True))

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Preprocessing
Encoded satisfaction as binary. Dropped ID columns. Encoded categoricals (gender, travel type, class).

## Feature Engineering
Using raw survey ratings and flight features. Ordinal ratings used directly.

## Baseline Model

In [ ]:
bl = LogisticRegression(max_iter=1000, random_state=SEED)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred):.4f}')

## LazyPredict Benchmark

In [ ]:
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

## FLAML AutoML

In [ ]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Accuracy={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

## Additional Models: CatBoost, LightGBM, XGBoost

In [ ]:
models = {}
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

xgb = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p, average='weighted'):.4f}")

## Top 3 Model Selection

In [ ]:
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

## Final Evaluation of Top 3

In [ ]:
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
    print(f"{name}: Acc={final_results[name]['Accuracy']:.4f}  F1={final_results[name]['F1']:.4f}")
    print(classification_report(y_test, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

## Error Analysis

In [ ]:
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    axes[1].text(0.5, 0.5, 'No feature importance', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

wrong = X_test[preds != y_test]
print(f'Correct: {(preds == y_test).sum()}, Wrong: {len(wrong)}, Error rate: {len(wrong)/len(y_test):.4f}')

## Interpretation & Business Insight
Online boarding, inflight wifi, and seat comfort are top satisfaction drivers. Airlines should prioritize digital experience and comfort.

## Limitations
- Survey data may have response bias
- Anonymized — can't link to actual passengers
- Service ratings are self-reported

## How to Improve
- Add NPS score correlation
- Segment by travel type (business vs personal)
- Time series of satisfaction trends

## Production Considerations
- Real-time post-flight survey scoring
- A/B testing service improvements
- Integration with CRM systems

## Common Mistakes
- Including ID as feature
- Not handling missing Arrival Delay
- Ignoring ordinal nature of ratings

## Mini Challenge
1. Compare satisfaction by travel class
2. Build separate models for business vs economy
3. Find the single most impactful service rating

## Final Summary
Predicted airline passenger satisfaction from survey data. Boosting models achieve >90% accuracy. Service quality ratings are the strongest predictors.

In [ ]:
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))